In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import json
import os
import time
import base64
import requests
from datetime import datetime

In [2]:
os.makedirs("data", exist_ok=True)
OUTPUT_FILE = "data/imdb_top250.json"
 
TOP250_URL  = "https://www.imdb.com/chart/top/"
BASE_URL    = "https://www.imdb.com"
 
# Headers para download da imagem
IMG_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )
}

In [3]:

# Inicializar driver
def create_driver() -> webdriver.Chrome:
    options = Options()
    options.add_argument("--lang=en-US")
    options.add_argument("--window-size=1920,1080")
    # Reduz logs desnecessários do Chrome no terminal
    options.add_experimental_option("excludeSwitches", ["enable-logging"])
    return webdriver.Chrome(options=options)
 
 

# Etapa 1 — lista do Top 250
def scrape_top250_list(driver: webdriver.Chrome) -> list[dict]:
    
    driver.get(TOP250_URL)
 
    # Aguarda os itens da lista carregarem
    WebDriverWait(driver, 15).until(
        EC.presence_of_all_elements_located(
            (By.CSS_SELECTOR, "li.ipc-metadata-list-summary-item")
        )
    )
 
    # Scroll até o fim para garantir que todos os 250 itens foram renderizados
    last_height = driver.execute_script("return document.body.scrollHeight")
    while True:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(1.5)
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            break
        last_height = new_height
 
    items = driver.find_elements(By.CSS_SELECTOR, "li.ipc-metadata-list-summary-item")
 
    movies = []
    for i, item in enumerate(items, start=1):
        # Título
        try:
            title = item.find_element(By.TAG_NAME, "h3").text
            # Remove o prefixo de ranking que o IMDb inclui no h3
            if ". " in title:
                title = title.split(". ", 1)[1]
        except Exception:
            title = None
 
        # Nota
        try:
            rating = float(
                item.find_element(By.CSS_SELECTOR, ".ipc-rating-star--rating").text
            )
        except Exception:
            rating = None
 
        # Metadados (ano, duração, classificação)
        try:
            metadata = item.find_elements(By.CSS_SELECTOR, ".cli-title-metadata-item")
            year = None
        except Exception:
            year = None
 
        # URL da página do filme
        try:
            link_tag = item.find_element(By.CSS_SELECTOR, "a[href*='/title/tt']")
            href = link_tag.get_attribute("href")
            # Remove parâmetros de query string
            movie_url = href.split("?")[0].rstrip("/")
        except Exception:
            movie_url = None
 
        movies.append({
            "rank":  i,
            "title": title,
            "year":  year,
            "imdb_rating": rating,
            "url":   movie_url,
        })
 
    print(f"  -> {len(movies)} filmes encontrados.\n")
    return movies
 

# Etapa 2 — detalhes de cada filme
def download_image_b64(url: str) -> str | None:
    """Baixa pôster e retorna como data URI base64."""
    if not url:
        return None
    try:
        resp = requests.get(url, headers=IMG_HEADERS, timeout=15)
        resp.raise_for_status()
        mime  = resp.headers.get("Content-Type", "image/jpeg").split(";")[0].strip()
        b64   = base64.b64encode(resp.content).decode("utf-8")
        return f"data:{mime};base64,{b64}"
    except Exception as exc:
        print(f"    [!] Falha ao baixar imagem: {exc}")
        return None
 
 
def scrape_movie_details(driver: webdriver.Chrome, movie: dict, index: int, total: int) -> dict:
    """Acessa a página do filme e extrai gêneros, diretores e pôster."""
    url   = movie.get("url")
    title = movie.get("title", "?")
    rank  = movie.get("rank", index)
 
    print(f"  [{index:>3}/{total}] #{rank:<3} {title[:48]:<48}", end=" ... ", flush=True)
 
    # Copia os dados já coletados na lista
    result = {**movie, "genres": [], "directors": [], "poster_url": None, "poster_b64": None}
 
    if not url:
        print("SEM URL")
        return result
 
    driver.get(url)
 
    try:
        WebDriverWait(driver, 12).until(
            EC.presence_of_element_located((By.TAG_NAME, "h1"))
        )
    except Exception:
        print("TIMEOUT")
        return result
    #Ano
    try:
        year_tag = driver.find_element(By.CSS_SELECTOR, "a[href*='releaseinfo']")
        result["year"] = int(year_tag.text.strip()[:4])

    except Exception:
        pass
    # Gêneros
    try:
        genre_tags = driver.find_elements(
        By.CSS_SELECTOR, "div[data-testid='interests'] a span"
        )
        result["genres"] = [g.text.strip() for g in genre_tags if g.text.strip()]
    except Exception:
        pass
 
    # Diretores — percorre os blocos de crédito e busca o label "Director"
    try:
        credit_blocks = driver.find_elements(
            By.CSS_SELECTOR, "li[data-testid='title-pc-principal-credit']"
        )
        for block in credit_blocks:
            try:
                label = block.find_element(
                    By.CSS_SELECTOR, "span.ipc-metadata-list-item__label"
                ).text
                if "Director" in label:
                    names = block.find_elements(By.TAG_NAME, "a")
                    result["directors"] = [n.text.strip() for n in names if n.text.strip()]
                    break
            except Exception:
                continue
    except Exception:
        pass
 
    # Pôster — tenta pegar a maior resolução via srcset
    try:
        poster_img = driver.find_element(
            By.CSS_SELECTOR, "div[data-testid='hero-media__poster'] img"
        )
        srcset = poster_img.get_attribute("srcset") or ""
        if srcset:
            urls = [
                s.strip().split(" ")[0]
                for s in srcset.split(",")
                if s.strip().startswith("http")
            ]
            result["poster_url"] = urls[-1] if urls else poster_img.get_attribute("src")
        else:
            result["poster_url"] = poster_img.get_attribute("src")
    except Exception:
        pass
 
    # Download da imagem do pôster
    result["poster_b64"] = download_image_b64(result["poster_url"])
 
    # Log resumido
    info = []
    if result["imdb_rating"]: info.append(f"{result['imdb_rating']}")
    if result["genres"]:      info.append("/".join(result["genres"][:2]))
    if result["directors"]:   info.append(result["directors"][0])
    print(", ".join(info) if info else "ok")
 
    return result
 

# Etapa 3 — salvar JSON
def save_json(data: list[dict], path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"\n  -> Arquivo salvo em: {path}")

In [4]:
def main():
    start = datetime.now()
    print(f"\nIMDb Top 250 Scraper  –  início: {start.strftime('%H:%M:%S')}\n")
 
    driver = create_driver()
 
    try:
        # Etapa 1
        print("Coletando top 250...")
        movies_list = scrape_top250_list(driver)
 
        # Etapa 2
        detailed = []
        total = len(movies_list)

 
        for i, movie in enumerate(movies_list, start=1):
            result = scrape_movie_details(driver, movie, i, total)
            detailed.append(result)
            time.sleep(0.5)   # pausa entre páginas
        
        print("Detalhes coletados")
 
    finally:
        driver.quit()
 
    # Etapa 3
    print("Salvando JSON")
    save_json(detailed, OUTPUT_FILE)
 
    elapsed  = (datetime.now() - start).total_seconds()
    ok       = sum(1 for m in detailed if m["imdb_rating"])
    no_genre = sum(1 for m in detailed if not m["genres"])
    no_dir   = sum(1 for m in detailed if not m["directors"])
    no_img   = sum(1 for m in detailed if not m["poster_b64"])
 
    print(f"""\n
    Total processados        : {total:>3}           
    Com nota IMDb            : {ok:>3}           
    Sem gêneros              : {no_genre:>3}           
    Sem diretores            : {no_dir:>3}           
    Sem imagem do pôster     : {no_img:>3}           
    Tempo total              : {elapsed:>7.1f}s
    """)
    
main()


IMDb Top 250 Scraper  –  início: 12:16:49

Coletando top 250...
  -> 250 filmes encontrados.

  [  1/250] #1   The Shawshank Redemption                         ... 9.3, Period Drama/Prison Drama, Frank Darabont
  [  2/250] #2   The Godfather                                    ... 9.2, Epic/Gangster, Francis Ford Coppola
  [  3/250] #3   The Dark Knight                                  ... 9.1, Action Epic/Epic, Christopher Nolan
  [  4/250] #4   The Godfather Part II                            ... 9.0, Epic/Gangster, Francis Ford Coppola
  [  5/250] #5   12 Angry Men                                     ... 9.0, Legal Drama/Psychological Drama, Sidney Lumet
  [  6/250] #6   The Lord of the Rings: The Return of the King    ... 9.0, Action Epic/Adventure Epic, Peter Jackson
  [  7/250] #7   Schindler's List                                 ... 9.0, Docudrama/Epic, Steven Spielberg
  [  8/250] #8   The Lord of the Rings: The Fellowship of the Rin ... 8.9, Action Epic/Adventure Epic, Peter 